# TS2 Skeletonization

## Setup

In [1]:
from mcf2swc import *

from swctools import SWCModel, FrustaSet, PointSet, plot_model

import logging
logging.basicConfig(level=logging.INFO)

import os

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Load Mesh and skeleton

In [2]:
spine_idx = 2
mcf_qst = 0.6
mcf_mcst = 5
obj_name = f"TS{spine_idx}"
skeleton_name = f"TS{spine_idx}_qst{mcf_qst}_mcst{mcf_mcst}"
mm = MeshManager(mesh_path=f"../data/mesh/processed/{obj_name}_simplified.obj")
raw_skeleton = SkeletonGraph.from_txt(f"../data/mcf_skeletons/{skeleton_name}.polylines.txt")
raw_skeleton.prune_short_branches_inplace(min_length=10)
print(raw_skeleton)
mm.visualize_mesh_3d(skel=raw_skeleton)

INFO:mcf2swc.mesh:Loaded mesh: 508 vertices, 1016 faces


SkeletonGraph with 118 nodes and 118 edges


# Skeleton optimization

In [3]:
from mcf2swc import (
    SkeletonGraph,
    MeshManager,
    SkeletonOptimizer,
    SkeletonOptimizerOptions,
)

do_skeleton_optimization = True

if do_skeleton_optimization:
    opts = SkeletonOptimizerOptions(
        max_iterations=5,
        step_size=5.0,
        smoothing_weight=0.1,
        verbose=True,
    )
    # Create optimizer and check for surface crossing
    optimizer = SkeletonOptimizer(raw_skeleton, mm.mesh, opts)
    has_crossing, num_outside, max_dist = optimizer.check_surface_crossing()
    print(f"Surface crossing: {has_crossing}")
    print(f"Points outside: {num_outside}/{raw_skeleton.total_points()}")
    print(f"Max distance: {max_dist:.4f}")
    # Run optimization
    print("\nOptimizing skeleton...")
    optimized_skeleton = optimizer.optimize()

    skeleton = optimized_skeleton
    all_skeletons = [raw_skeleton, optimized_skeleton]
else:
    skeleton = raw_skeleton
    all_skeletons = [raw_skeleton]


mm.visualize_mesh_3d(skel=all_skeletons, skel_color=["crimson", "blue"])


INFO:mcf2swc.skeleton_optimizer:No surface crossing detected - all nodes inside mesh
INFO:mcf2swc.skeleton_optimizer:No surface crossing detected - all nodes inside mesh
INFO:mcf2swc.skeleton_optimizer:Starting skeleton optimization...
INFO:mcf2swc.skeleton_optimizer:  Nodes: 118
INFO:mcf2swc.skeleton_optimizer:  Max iterations: 5
INFO:mcf2swc.skeleton_optimizer:  Step size: 5.0000
INFO:mcf2swc.skeleton_optimizer:  Smoothing weight: 0.1000


Surface crossing: False
Points outside: 0/118
Max distance: 0.0000

Optimizing skeleton...


INFO:mcf2swc.skeleton_optimizer:  Iteration 0: avg movement = 4.831948
INFO:mcf2swc.skeleton_optimizer:Optimization complete


# Fit morphology

In [9]:
spacing = 70

swc_out_dir = f"../data/swc/current/{skeleton_name}"

# check if directory exists, if not create it
if not os.path.exists(swc_out_dir):
    os.makedirs(swc_out_dir)

radius_strategy_list = [
    "equivalent_area",
    # "equivalent_perimeter",
    # "section_median",
    # "section_circle_fit",
    # "nearest_surface",
]
for radius_strategy in radius_strategy_list:
    print(f"Computing skeleton for radius_strategy={radius_strategy} ...", end="")
    morph = fit_morphology(
        mm.mesh, 
        skeleton, 
        options=FitOptions(
        spacing=spacing,
        radius_strategy=radius_strategy,
        snap_polylines_to_mesh=False)
)
    # write swc to file
    morph.to_swc_file(f"{swc_out_dir}/TS{spine_idx}_s{spacing}_{radius_strategy}.swc")
    morph.print_attributes(node_info=False, edge_info=False)
    print("DONE")

INFO:mcf2swc.graph_fitting:Tracing start: mesh[V=508,F=1016], skeleton[nodes=118,edges=118], spacing=70, radius_strategy=equivalent_area
INFO:mcf2swc.graph_fitting:Computing skeleton node radii using multi-tangent approach...


Computing skeleton for radius_strategy=equivalent_area ...

INFO:mcf2swc.graph_fitting:Computed radii for 118 skeleton nodes
INFO:mcf2swc.graph_fitting:Edge group 0: input_pts=62 -> samples=6
INFO:mcf2swc.graph_fitting:Edge group 2: input_pts=4 -> samples=2
INFO:mcf2swc.graph_fitting:Edge group 3: input_pts=16 -> samples=2
INFO:mcf2swc.graph_fitting:Edge group 4: input_pts=20 -> samples=4
INFO:mcf2swc.graph_fitting:Edge group 5: input_pts=21 -> samples=4
INFO:mcf2swc.graph_fitting:Tracing done: nodes=13, edges=13, samples=18, section=0, fallback=0


MorphologyGraph: nodes=13, edges=13, components=1, cycles=1, branch_points=2, leaves=2, self_loops=0, density=0.1667
DONE


# Plotting

In [10]:
# plot using swctools
make_html = False

for radius_strategy in radius_strategy_list:
    swc_filepath = f"{swc_out_dir}/TS{spine_idx}_s{spacing}_{radius_strategy}.swc"
    model = SWCModel.from_swc_file(swc_filepath)
    model.print_attributes(node_info=False, edge_info=False)
    frusta = FrustaSet.from_swc_model(model)
    title = f"TS{spine_idx}_s{spacing}_{radius_strategy}"
    fig = plot_model(swc_model=model, frusta=frusta, slider=True, title=title)
    fig.show()
    if make_html:
        fig.write_html(f"../plotly/TS{spine_idx}_s{spacing}_{radius_strategy}.html")


INFO:swctools.io:parse_swc start strict=True validate_reconnections=True float_tol=1e-09
INFO:swctools.io:parse_swc done records=14 reconnections=1 header=5
INFO:swctools.model:SWCModel.from_parse_result records=14 reconnections=1 header=5
INFO:swctools.model:SWCModel.from_swc_file built nodes=14 edges=13 strict=True validate_reconnections=True
INFO:swctools.geometry:batch_frusta count=13 sides=16 end_caps=False verts=416 faces=416
INFO:swctools.geometry:FrustaSet.from_swc_model edges=13 sides=16 end_caps=False
INFO:swctools.geometry:batch_frusta count=13 sides=16 end_caps=False verts=416 faces=416
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.0
INFO:swctools.geometry:batch_frusta count=13 sides=16 end_caps=False verts=416 faces=416
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.05
INFO:swctools.geometry:batch_frusta count=13 sides=16 end_caps=False verts=416 faces=416
INFO:swctools.geometry:FrustaSet.scaled radius_scale=0.1
INFO:swctools.geometry:batch_frusta count=13

SWCModel: nodes=14, edges=13, components=1, cycles=0, branch_points=1, roots=1, leaves=3, self_loops=0, density=0.1429


INFO:swctools.viz:plot_model slider=True segments=13 radius_scale_range=[0.0,1.0]
